In [14]:
import pandas as pd

raw_data = pd.read_csv('../datasets/data_EUR_raw.csv', na_values=['-'])

Check the data type of each column in the raw data.

In [15]:
raw_data.dtypes

Publication                       int64
No. of obs                        int64
Temp                            float64
Prec                            float64
Clay                              int64
CEC                             float64
BD                              float64
pH                              float64
ST                              float64
WFPS                              int64
SOC                             float64
TN                              float64
C/N                             float64
NH4+-N                          float64
NO3_-N                          float64
MN                              float64
Crop type                        object
crop_class                       object
Sowing date                      object
Harvest date                     object
Irrigation (date)                object
Pesticides/Herbicides (date)     object
Fertilizer N type                object
fertilization_code               object
fertilization_class              object


Divide the variables into different groups

In [16]:
numeric_static_features = {
    'Clay': '粘土含量',
    'CEC': '阳离子交换容量',
    'BD': '土壤容重',
    'pH': 'pH',
    'SOC': '有机碳含量',
    'TN': '总氮含量',
    'C/N': '碳氮比',
}

numeric_dynamic_features = {
    'Temp': '温度',
    'Prec': '降水量',
    'ST': '土壤温度',
    'WFPS': '土壤含水量',
    'NH4+-N': '铵态氮',
    'NO3_-N': '硝态氮',
    'MN': '矿质氮含量（铵态+硝态氮）'
}

classification_static_features = {
    'crop_class': '作物类型',
}

fertilization_features = {
    'fertilization_class': '上次施肥类型',
    'Split N amount': '上次施肥量',
    'appl_class': '上次施肥方式',
    'ferdur': '该次测量距上次施肥的天数'
}

optional_features = {
    'NH4+-N': '铵态氮',
    'NO3_-N': '硝态氮',
    'MN': '矿质氮含量（铵态+硝态氮）'
}

auxiliary_variables = {
    'Sowing date': '播种日期',
    'Harvest date': '收获日期',
    'Fertilization date': '施肥日期',
    'Emission date': '排放日期(观测日期)',
}

group_variables = {
    'No. of obs': 'ID',
    'Publication': '文献编号',
    'control_group': '组号',
    'sowdur': '该次测量到播种之间的天数',
}

drop_variables = {
    'Crop type',
    'Irrigation (date)',
    'Pesticides/Herbicides (date)',
    'Fertilizer N type',
    'fertilization_code',
    'inhibitor',
    'Total N amount',
    'Appl.code',
    'SE',
    'Duration'
}

labels = {
    'Daily fluxes': 'N2O排放通量',
}

Make sure our group can cover all the columns.

In [17]:
len(set(raw_data.columns) - numeric_dynamic_features.keys() - numeric_static_features.keys() - fertilization_features.keys() - classification_static_features.keys() - group_variables.keys() - drop_variables - auxiliary_variables.keys() - labels.keys()) == 0

True

Remove redundant columns, adjust the column order and delete all samples from the fallow period.

In [18]:
processed_data = raw_data.drop(columns=drop_variables)
processed_data = processed_data[[
    *group_variables.keys(),
    *labels.keys(),
    *classification_static_features.keys(),
    *numeric_static_features.keys(),
    *fertilization_features.keys(),
    *numeric_dynamic_features.keys(),
]]

print('Before drop all samples of fallow period, total sample: ', len(processed_data))
fallow_mask = (processed_data['sowdur'] < 0) | (processed_data['control_group'] < 0)
print('Fallow samples: ', fallow_mask.sum())
processed_data = processed_data[~fallow_mask]
print('After delete all samples of fallow period, processed sample: ', len(processed_data))

Before drop all samples of fallow period, total sample:  48688
Fallow samples:  12578
After delete all samples of fallow period, processed sample:  36110


For samples where `Split N amount` is greater than 0 but `ferdur` is less than zero, reset `ferdur` to 0 to facilitate model processing.

In [19]:
processed_data.loc[(~processed_data['Split N amount'].isnull()) & (processed_data['ferdur'] < 0), 'ferdur'] = 0

Fill the missing parts of `Split N amount` with zeros.

In [20]:
processed_data['Split N amount'] = processed_data['Split N amount'].fillna(0)

Check the statistical characteristics and missing proportion of numeric types.

In [21]:
numeric_columns = labels.keys() | numeric_static_features.keys() | numeric_dynamic_features.keys() | {'ferdur', 'Split N amount'}

numeric_df = processed_data[list(numeric_columns)]
stats = numeric_df.describe().transpose()

missing_ratio = numeric_df.isnull().sum() / len(numeric_df) * 100
missing_df = missing_ratio.to_frame(name='missing_ratio (%)')

numeric_summary = stats.join(missing_df).round(3)
numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing_ratio (%)
NH4+-N,13763.0,8.482,16.634,0.00,1.19,2.790,8.365,357.93,61.886
MN,13910.0,22.968,31.862,0.00,5.08,12.615,29.610,700.00,61.479
TN,24144.0,1.684,2.535,0.20,1.00,1.300,1.600,33.60,33.138
pH,36110.0,6.993,0.844,4.00,6.40,6.900,7.700,8.60,0.000
ferdur,36110.0,35.180,50.417,-1.00,-1.00,15.000,52.000,367.00,0.000
Clay,36110.0,22.392,11.462,2.00,13.00,21.000,28.000,64.00,0.000
SOC,36110.0,18.691,30.986,2.20,9.30,12.900,18.200,412.00,0.000
ST,36110.0,13.311,7.135,-11.90,8.30,13.100,18.000,51.10,0.000
Temp,36110.0,12.663,6.750,-16.30,8.10,12.600,17.200,31.40,0.000
Prec,36110.0,2.456,6.819,0.00,0.00,0.000,1.800,262.50,0.000


Ensure that the `Split N amount` column only has missing values when `ferdur < 0`.

In [22]:
processed_data[(processed_data['Split N amount'].isnull()) & (processed_data['ferdur'] >= 0)]

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN


In [23]:
processed_data[(~processed_data['Split N amount'].isnull()) & (processed_data['ferdur'] < 0)]

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN
0,1,1,1,0,-0.52,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,18.8,0.0,23.6,41,1.46,2.66,4.12
1,2,1,2,0,0.17,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,18.8,0.0,23.6,39,1.82,2.78,4.60
2,3,1,3,0,-0.48,fruit,28,16.0,1.4,7.6,...,0.0,deep,-1,18.8,0.0,23.6,41,1.24,1.83,3.07
3,4,1,4,0,0.18,fruit,28,16.0,1.4,7.6,...,0.0,deep,-1,18.8,0.0,23.6,39,1.82,2.55,4.38
4,5,1,5,0,-0.35,fruit,28,16.0,1.4,7.6,...,0.0,deep,-1,18.8,0.0,23.6,41,1.09,3.05,4.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48490,51798,220,1,67,-0.76,legume,23,21.0,1.2,5.4,...,0.0,no,-1,23.9,0.0,26.8,40,NaN,NaN,NaN
48491,51799,220,1,68,-6.07,legume,23,21.0,1.2,5.4,...,0.0,no,-1,25.8,0.0,28.3,39,NaN,NaN,NaN
48492,51800,220,1,69,-3.60,legume,23,21.0,1.2,5.4,...,0.0,no,-1,24.7,2.5,28.3,39,NaN,NaN,NaN
48493,51801,220,1,70,2.82,legume,23,21.0,1.2,5.4,...,0.0,no,-1,22.1,1.1,25.0,38,NaN,NaN,NaN


Check the statistical characteristics and missing proportion of classification variables.

In [24]:
classification_columns = classification_static_features.keys() | (fertilization_features.keys() - {'ferdur', 'Split N amount'})

classification_df = processed_data[list(classification_columns)]
stats = classification_df.describe().transpose()

missing_ratio = classification_df.isnull().sum() / len(classification_df) * 100
missing_df = missing_ratio.to_frame(name='missing_ratio (%)')

classification_summary = stats.join(missing_df).round(3)
classification_summary


,count,unique,top,freq,missing_ratio (%)
appl_class,36110,3,surface,22665,0.0
crop_class,36110,11,wheat,11004,0.0
fertilization_class,36110,9,AN,8239,0.0


Sort according to the variables in `group_variables` and save the processed variables to `../datasets/data_EUR_reordered.csv`.

In [25]:
reordered_data = processed_data.sort_values(
    by=['Publication', 'control_group', 'sowdur'],
    ascending=True
)
reordered_data.to_csv('../datasets/data_EUR_reordered.csv', index=False)
reordered_data

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN
0,1,1,1,0,-0.52,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,18.8,0.0,23.6,41,1.46,2.66,4.12
6,7,1,1,20,0.87,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,17.9,0.0,22.1,60,2.40,3.75,6.16
12,13,1,1,22,0.15,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,19.5,0.0,22.0,41,0.95,2.66,3.61
18,19,1,1,27,0.26,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,21.5,0.0,23.5,60,1.82,5.21,7.03
24,25,1,1,29,0.39,fruit,28,16.0,1.4,7.6,...,0.0,no,-1,23.8,0.0,25.8,59,2.48,4.36,6.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48636,51944,220,2,124,2.56,maize,23,21.0,1.2,5.4,...,92.0,surface,111,17.6,0.0,19.8,53,NaN,NaN,NaN
48637,51945,220,2,125,6.06,maize,23,21.0,1.2,5.4,...,92.0,surface,112,17.9,0.0,20.2,52,NaN,NaN,NaN
48638,51946,220,2,126,9.26,maize,23,21.0,1.2,5.4,...,92.0,surface,113,18.2,0.0,20.2,51,NaN,NaN,NaN
48639,51947,220,2,127,1.84,maize,23,21.0,1.2,5.4,...,92.0,surface,114,18.3,0.0,20.5,50,NaN,NaN,NaN


In [26]:
reordered_data.dtypes

No. of obs               int64
Publication              int64
control_group            int64
sowdur                   int64
Daily fluxes           float64
crop_class              object
Clay                     int64
CEC                    float64
BD                     float64
pH                     float64
SOC                    float64
TN                     float64
C/N                    float64
fertilization_class     object
Split N amount         float64
appl_class              object
ferdur                   int64
Temp                   float64
Prec                   float64
ST                     float64
WFPS                     int64
NH4+-N                 float64
NO3_-N                 float64
MN                     float64
dtype: object